# SmileAlign v2, YOLOv8n-seg campuran intraoral + AlphaDent (1 kelas `tooth`)

Perbaikan atas v1: dataset sanjungan dua domain.
- **Intraoral Tooth Detection** (Roboflow, CC BY 4.0) yang sudah kamu upload ke Drive: `smilealign/intraoral_yolov8.zip`
- **AlphaDent** (Apache-2.0, diunduh langsung dari HuggingFace di Colab) 1.455 foto gigi DSLR

Semua kelas disatukan ke `tooth` (backend hanya pakai geometri instance).

## Cara pakai
1. Pastikan `intraoral_yolov8.zip` ada di `MyDrive/smilealign/` (sudah pernah).
2. Opsional tapi bagus: taruh `backend/models/seg/best.pt` v1 ke Drive `smilealign/best.pt`
   sebagai inisialisasi (warm start).
3. Runtime > Change runtime type > **GPU T4**.
4. Run All. Latihan ±1 jam (imgsz 960). Hasil di `smilealign/runs-smilealign-seg-v2/weights/best.pt`.
5. Unduh best.pt kemudian taruh di `backend/models/seg/best.pt` di komputer (REWIND mode).

In [ ]:
# Sel 0: mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Sel 1: ekstrak intraoral (zip Windows-style) + unduh dan ekstrak AlphaDent dari HuggingFace
import os, zipfile
from pathlib import Path

INTRAORAL_ZIP = '/content/drive/MyDrive/smilealign/intraoral_yolov8.zip'
INTRAORAL_DIR = '/content/datasets/intraoral'
ALPHA_DIR = '/content/datasets/alphadent'
MIX_DIR = Path('/content/datasets/mix_v2')

os.makedirs(INTRAORAL_DIR, exist_ok=True)
with zipfile.ZipFile(INTRAORAL_ZIP) as z:
    for info in z.infolist():
        name = info.filename.replace('\\', '/')
        target = os.path.join(INTRAORAL_DIR, name)
        if name.endswith('/'):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            dst.write(src.read())
print('intraoral OK:', os.listdir(os.path.join(INTRAORAL_DIR, 'intraoral')))

from huggingface_hub import hf_hub_download
zip_path = hf_hub_download(
    repo_id='ZFTurbo/AlphaDent',
    filename='AlphaDent.zip',
    repo_type='dataset',
    local_dir='/content/ad_zip',
)
print('AlphaDent zip:', zip_path)
os.makedirs(ALPHA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(ALPHA_DIR)

# cari folder yang memuat images/ dan labels/
AD_ROOT = None
for root, dirs, files in os.walk(ALPHA_DIR):
    if 'images' in dirs and 'labels' in dirs:
        AD_ROOT = root
        break
assert AD_ROOT, 'Struktur AlphaDent tidak ketemu; cek isi zip'

# fase: struktur yang didukung {images,labels}/{train,valid,test} ATAU {train,...}/{images,labels}
def has_struct(root):
    a = os.path.isdir(os.path.join(root, 'images', 'train')) and os.path.isdir(os.path.join(root, 'labels', 'train'))
    b = os.path.isdir(os.path.join(root, 'train', 'images')) and os.path.isdir(os.path.join(root, 'train', 'labels'))
    return bool(a or b)

if not has_struct(AD_ROOT):
    for root, dirs, files in os.walk(ALPHA_DIR):
        if has_struct(root):
            AD_ROOT = root
            break
print('AD_ROOT:', AD_ROOT)
for sub in ('images', 'labels'):
    print(' ', sub, '->', os.listdir(os.path.join(AD_ROOT, sub))[:3])

In [ ]:
# Sel 2: gabungkan jadi 1 kelas (logika sama dengan scripts/prepare_v2_mix.py)
import shutil
from pathlib import Path

JOBS = [
    ('io_', Path(INTRAORAL_DIR) / 'intraoral'),   # struktur {split}/{images,labels}
    ('ad_', Path(AD_ROOT)),
]
SPLITS = [('train', 'train'), ('valid', 'valid'), ('test', 'valid')]


def pick_dirs(root, split):
    imgs = [p for p in (root / split / 'images', root / 'images' / split) if p.is_dir()]
    lbls = [p for p in (root / split / 'labels', root / 'labels' / split) if p.is_dir()]
    return (imgs[0] if imgs else None), (lbls[0] if lbls else None)


def normalize_row(line):
    """Samakan baris label ke format segmentasi, kelas selalu 0.

    AlphaDent mencampur baris polygon dan bbox; Ultralytics menolak file yang
    mencampur keduanya, jadi bbox diubah menjadi polygon persegi empat titik.
    """
    parts = line.split()
    if len(parts) < 2:
        return None
    coords = parts[1:]
    if len(coords) == 4:
        cx, cy, bw, bh = (float(v) for v in coords)
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return '0 ' + ' '.join(f'{v:.6f}' for pt in corners for v in pt)
    if len(coords) >= 6 and len(coords) % 2 == 0:
        return '0 ' + ' '.join(coords)
    return None


counts = {'train': 0, 'valid': 0}
dropped = 0
for prefix, root in JOBS:
    for split_src, split_dst in SPLITS:
        imgs, lbls = pick_dirs(root, split_src)
        if imgs is None or lbls is None:
            continue
        (MIX_DIR / split_dst / 'images').mkdir(parents=True, exist_ok=True)
        (MIX_DIR / split_dst / 'labels').mkdir(parents=True, exist_ok=True)
        for img in sorted(imgs.iterdir()):
            if img.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            lbl = lbls / (img.stem + '.txt')
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                t = line.strip()
                if not t:
                    continue
                row = normalize_row(t)
                if row is None:
                    dropped += 1
                    continue
                rows.append(row)
            if not rows:
                continue
            shutil.copy(img, MIX_DIR / split_dst / 'images' / f'{prefix}{img.name}')
            (MIX_DIR / split_dst / 'labels' / f'{prefix}{img.stem}.txt').write_text('\n'.join(rows) + '\n')
            counts[split_dst] += 1

(MIX_DIR / 'data.yaml').write_text('train: train/images\nval: valid/images\nnc: 1\nnames: [\'tooth\']\n')
print('merged:', counts, '->', MIX_DIR)
if dropped:
    print('baris label dibuang (format tidak dikenali):', dropped)


In [ ]:
# Sel 3: setup ultralytics
!pip -q install ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
# Sel 4: latihan v2 (imgsz 960, warm-start dari bobot v1 bila ada)
from pathlib import Path
from ultralytics import YOLO

V1 = '/content/drive/MyDrive/smilealign/best.pt'
init = V1 if os.path.exists(V1) else 'yolov8n-seg.pt'
print('Init dari:', init)

model = YOLO(init)
results = model.train(
    data=str(MIX_DIR / 'data.yaml'),
    epochs=120,
    imgsz=960,
    batch=12,
    device=0,
    project='/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2',
    name='train',
    patience=20,
    cache=False,
    workers=2,
)
metrics = model.val()
print(metrics.results_dict)

Selesai. Hasil ada di Drive, folder `smilealign/runs-smilealign-seg-v2/`.
Namanya `train`, tetapi bila folder itu sudah ada Ultralytics memakai `train-2`, `train-3`, dan seterusnya.
Cek folder terbaru, lalu unduh `weights/best.pt` dan taruh di `backend/models/seg/best.pt`.

Bila sesi Colab terputus, lanjutkan tanpa mengulang dari nol:
```python
from ultralytics import YOLO
model = YOLO('<folder-terbaru>/weights/last.pt')
model.train(resume=True)
```

Referensi sitasi: AlphaDent (Sosnin dkk., 2025, Apache-2.0) + Intraoral Tooth Detection (CC BY 4.0).